# 7. Multi-Modal Integration & Clinical Ablation Study

**Goal:** In Notebook `05b`, we finalized 8 predictive pipelines utilizing strictly microbiome data. In this notebook, we test the hypothesis that integrating clinical metadata will enhance the models' predictive performance.

**Methodology:**
To understand exactly *which* metadata drives performance, we will perform a strict ablation study. For every one of our 8 baseline models, we will train 3 new variations:
1. **+ Age Only**
2. **+ Gender Only**
3. **+ Age & Gender (Both)**

We utilize Scikit-Learn's `ColumnTransformer` to route the data. Microbiome features undergo their fixed dimensionality reduction (KW $p < 0.01$ or PCA $60\%$), while clinical features bypass reduction but undergo standard scaling. We strictly mirror the `n_iter=50` parameter search from Notebook 05b to ensure fair comparisons.

In [1]:
# 1. Imports and Data Preparation
import pandas as pd
import numpy as np
import joblib
import os
import warnings
from scipy.stats import kruskal

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV, cross_validate

warnings.filterwarnings("ignore")
os.makedirs('../results', exist_ok=True)

print("Libraries loaded.")

# Load Data
meta = pd.read_csv("../data/processed/metadata_final.tsv", sep="\t", index_col="Sample")
genera_clr = pd.read_csv("../data/processed/genera_clr.tsv", sep="\t", index_col=0)
meta = meta.loc[genera_clr.index]

# Extract and Clean Metadata (Age & Gender)
meta_subset = meta[["consent_age", "Gender"]].copy()
meta_subset["consent_age"] = meta_subset["consent_age"].fillna(meta_subset["consent_age"].median())
meta_subset["Gender"] = meta_subset["Gender"].map({"Male": 0, "Female": 1})
meta_subset["Gender"] = meta_subset["Gender"].fillna(meta_subset["Gender"].mode()[0])

# Combine everything into one master DataFrame
X_all = pd.concat([genera_clr, meta_subset], axis=1, join='inner')

# Define column groups
microbe_cols = genera_clr.columns.tolist()

# Create binary datasets
def make_binary_dataset(meta, features, group_a, group_b):
    mask = meta["Study.Group"].isin([group_a, group_b])
    meta_sub = meta[mask]
    X = features.loc[meta_sub.index]
    y = (meta_sub["Study.Group"] == group_a).astype(int)
    return X, y

X_uc, y_uc = make_binary_dataset(meta, X_all, "UC", "nonIBD")
X_cd, y_cd = make_binary_dataset(meta, X_all, "CD", "nonIBD")

print(f"Data Prep Complete. UC: {X_uc.shape[0]} samples | CD: {X_cd.shape[0]} samples")

Libraries loaded.
Data Prep Complete. UC: 56 samples | CD: 75 samples


### 2. Custom Feature Selector & Evaluation Function

In [2]:
# Custom Kruskal-Wallis Selector (Fixed p < 0.01)
class KruskalSelector(BaseEstimator, TransformerMixin):
    def __init__(self, p_threshold=0.01):
        self.p_threshold = p_threshold
        self.selected_indices_ = None
        
    def fit(self, X, y):
        p_values = []
        X_arr = X.values if isinstance(X, pd.DataFrame) else X
        y_arr = y.values if isinstance(y, pd.Series) else y
        
        for i in range(X_arr.shape[1]):
            group0 = X_arr[:, i][y_arr == 0]
            group1 = X_arr[:, i][y_arr == 1]
            stat, p = kruskal(group0, group1)
            p_values.append(p)
            
        p_series = pd.Series(p_values)
        self.selected_indices_ = np.where(p_series < self.p_threshold)[0]
        
        if len(self.selected_indices_) == 0:
            self.selected_indices_ = np.argsort(p_values)[:5]
            
        return self
        
    def transform(self, X):
        X_arr = X.values if isinstance(X, pd.DataFrame) else X
        return X_arr[:, self.selected_indices_]

# Robust 3-Run Evaluation Function
def evaluate_multiple_runs(pipeline, X, y, model_name, n_runs=3, n_splits=5):
    metrics = {'roc_auc': [], 'accuracy': [], 'f1': [], 'precision': [], 'recall': []}
    
    for run in range(n_runs):
        cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42 + run)
        scores = cross_validate(pipeline, X, y, cv=cv, 
                                scoring=['roc_auc', 'accuracy', 'f1', 'precision', 'recall'], 
                                n_jobs=-1)
        for metric in metrics.keys():
            metrics[metric].extend(scores[f'test_{metric}'])
            
    summary = {'Model Configuration': model_name}
    for metric, values in metrics.items():
        clean_name = metric.replace("test_", "").replace("roc_auc", "AUC").capitalize()
        if clean_name == "Roc_auc": clean_name = "AUC"
        if clean_name == "F1": clean_name = "F1-Score"
        
        summary[clean_name] = f"{np.mean(values):.3f} ± {np.std(values):.3f}"
        if metric == 'roc_auc':
            summary['_sort_auc'] = np.mean(values)
            
    return summary

### 3. Multi-Modal Pipeline Engine
This function dynamically accepts a list of metadata columns (`meta_cols`). It constructs the pipeline to only inject the specific metadata requested, allowing us to perform our ablation study.

*Note: Hyperparameter grids and `n_iter=50` match Notebook 05b exactly.*

In [3]:
# --- Exact Hyperparameter Grids from 05b ---
svm_param_grid = {
    'clf__C': np.logspace(-3, 3, 15),
    'clf__kernel': ['linear', 'rbf', 'poly', 'sigmoid'],
    'clf__gamma': ['scale', 'auto', 0.0001, 0.001, 0.01, 0.1, 1],
    'clf__degree': [2, 3],
    'clf__coef0': [0.0, 0.5, 1.0]
}

logreg_param_grid = {
    'clf__C': np.logspace(-4, 4, 20),
    'clf__penalty': ['l1', 'l2'],
    'clf__solver': ['liblinear', 'saga'], 
    'clf__max_iter': [2000]
}

def build_and_tune_multimodal(X, y, task_name, selector_type, clf_type, meta_cols, tag):
    print(f"Training: {task_name} | {selector_type} | {clf_type} | {tag}")
    
    # 1. Define the Microbiome Path
    if selector_type == "KW":
        microbe_path = Pipeline([
            ('kw', KruskalSelector(p_threshold=0.01)),
            ('scaler', StandardScaler())
        ])
    elif selector_type == "PCA":
        microbe_path = Pipeline([
            ('scaler', StandardScaler()),
            ('pca', PCA(n_components=0.60, svd_solver='full', random_state=42))
        ])

    # 2. The Traffic Cop (Combines Microbes + specific metadata)
    preprocessor = ColumnTransformer(
        transformers=[
            ('microbes', microbe_path, microbe_cols),
            ('clinical', StandardScaler(), meta_cols) # Uses only the requested metadata columns
        ]
    )

    # 3. Define Classifier Step
    if clf_type == "SVM":
        clf_step = ('clf', SVC(probability=True, random_state=42, class_weight='balanced'))
        param_grid = svm_param_grid
    elif clf_type == "LogReg":
        clf_step = ('clf', LogisticRegression(random_state=42, class_weight='balanced'))
        param_grid = logreg_param_grid

    # 4. Assemble Final Pipeline
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        clf_step
    ])

    # 5. Grid Search (50 iterations as per 05b)
    search = RandomizedSearchCV(
        pipeline, param_distributions=param_grid, 
        n_iter=50, scoring='roc_auc', 
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42), 
        n_jobs=-1, random_state=42
    )
    
    search.fit(X, y)
    best_model = search.best_estimator_
    
    # Save to disk with the specific tag
    file_name = f"../results/final_{task_name.lower().split()[0]}_{selector_type.lower()}_{clf_type.lower()}_{tag.lower().replace('+', '')}.pkl"
    joblib.dump(best_model, file_name)
    
    return best_model

### 4. Execute Multi-Modal Training (24 Variations)

In [4]:
print("=== Starting Multi-Modal Ablation Study ===")

combined_models = {}

# Define the 3 metadata combinations
ablation_tests = {
    "+Age": ["consent_age"],
    "+Gender": ["Gender"],
    "+Both": ["consent_age", "Gender"]
}

# The 4 base configurations
base_configs = [
    ("KW", "SVM"), ("KW", "LogReg"), 
    ("PCA", "SVM"), ("PCA", "LogReg")
]

# Train UC Models
print("\n--- Processing UC vs nonIBD ---")
for sel, clf in base_configs:
    for tag, cols in ablation_tests.items():
        name = f"UC_{sel}_{clf}_{tag}"
        combined_models[name] = build_and_tune_multimodal(X_uc, y_uc, "UC vs nonIBD", sel, clf, cols, tag)

# Train CD Models
print("\n--- Processing CD vs nonIBD ---")
for sel, clf in base_configs:
    for tag, cols in ablation_tests.items():
        name = f"CD_{sel}_{clf}_{tag}"
        combined_models[name] = build_and_tune_multimodal(X_cd, y_cd, "CD vs nonIBD", sel, clf, cols, tag)

print("\nAll 24 Multi-Modal variations tuned and saved to disk.")

=== Starting Multi-Modal Ablation Study ===

--- Processing UC vs nonIBD ---
Training: UC vs nonIBD | KW | SVM | +Age
Training: UC vs nonIBD | KW | SVM | +Gender
Training: UC vs nonIBD | KW | SVM | +Both
Training: UC vs nonIBD | KW | LogReg | +Age
Training: UC vs nonIBD | KW | LogReg | +Gender
Training: UC vs nonIBD | KW | LogReg | +Both
Training: UC vs nonIBD | PCA | SVM | +Age
Training: UC vs nonIBD | PCA | SVM | +Gender
Training: UC vs nonIBD | PCA | SVM | +Both
Training: UC vs nonIBD | PCA | LogReg | +Age
Training: UC vs nonIBD | PCA | LogReg | +Gender
Training: UC vs nonIBD | PCA | LogReg | +Both

--- Processing CD vs nonIBD ---
Training: CD vs nonIBD | KW | SVM | +Age
Training: CD vs nonIBD | KW | SVM | +Gender
Training: CD vs nonIBD | KW | SVM | +Both
Training: CD vs nonIBD | KW | LogReg | +Age
Training: CD vs nonIBD | KW | LogReg | +Gender
Training: CD vs nonIBD | KW | LogReg | +Both
Training: CD vs nonIBD | PCA | SVM | +Age
Training: CD vs nonIBD | PCA | SVM | +Gender
Training

### 5. Multi-Modal Robustness Evaluation & Leaderboard
We pass all 24 updated pipelines through our 15-fold cross-validation engine.

In [5]:
print("=== Running Multi-Modal Robustness Evaluation (15 folds per model) ===")

results_list = []

for config_name, model in combined_models.items():
    X_target, y_target = (X_uc, y_uc) if "UC" in config_name else (X_cd, y_cd)
    summary = evaluate_multiple_runs(model, X_target, y_target, config_name)
    results_list.append(summary)

df_final_combined = pd.DataFrame(results_list)

df_uc_combined = df_final_combined[df_final_combined['Model Configuration'].str.contains("UC")].sort_values('_sort_auc', ascending=False).drop(columns=['_sort_auc'])
df_cd_combined = df_final_combined[df_final_combined['Model Configuration'].str.contains("CD")].sort_values('_sort_auc', ascending=False).drop(columns=['_sort_auc'])

print("\n--- ABLATION LEADERBOARD: UC vs nonIBD (+Metadata) ---")
display(df_uc_combined.reset_index(drop=True))

print("\n--- ABLATION LEADERBOARD: CD vs nonIBD (+Metadata) ---")
display(df_cd_combined.reset_index(drop=True))

=== Running Multi-Modal Robustness Evaluation (15 folds per model) ===

--- ABLATION LEADERBOARD: UC vs nonIBD (+Metadata) ---


,Model Configuration,Auc,Accuracy,F1-Score,Precision,Recall
0,UC_PCA_SVM_+Age,0.591 ± 0.148,0.540 ± 0.151,0.491 ± 0.191,0.580 ± 0.145,0.456 ± 0.247
1,UC_PCA_SVM_+Gender,0.588 ± 0.169,0.565 ± 0.162,0.543 ± 0.190,0.625 ± 0.205,0.511 ± 0.215
2,UC_KW_SVM_+Both,0.567 ± 0.168,0.542 ± 0.045,0.699 ± 0.031,0.541 ± 0.030,0.989 ± 0.042
3,UC_PCA_LogReg_+Gender,0.552 ± 0.158,0.549 ± 0.117,0.538 ± 0.167,0.586 ± 0.156,0.533 ± 0.229
4,UC_PCA_LogReg_+Both,0.552 ± 0.158,0.549 ± 0.117,0.538 ± 0.167,0.586 ± 0.156,0.533 ± 0.229
5,UC_PCA_LogReg_+Age,0.552 ± 0.158,0.549 ± 0.117,0.538 ± 0.167,0.586 ± 0.156,0.533 ± 0.229
6,UC_PCA_SVM_+Both,0.509 ± 0.201,0.536 ± 0.018,0.565 ± 0.282,0.436 ± 0.218,0.800 ± 0.400
7,UC_KW_SVM_+Gender,0.494 ± 0.164,0.489 ± 0.152,0.512 ± 0.146,0.532 ± 0.144,0.511 ± 0.177
8,UC_KW_LogReg_+Gender,0.477 ± 0.181,0.490 ± 0.130,0.512 ± 0.146,0.526 ± 0.137,0.522 ± 0.191
9,UC_KW_LogReg_+Age,0.477 ± 0.181,0.490 ± 0.130,0.512 ± 0.146,0.526 ± 0.137,0.522 ± 0.191



--- ABLATION LEADERBOARD: CD vs nonIBD (+Metadata) ---


,Model Configuration,Auc,Accuracy,F1-Score,Precision,Recall
0,CD_PCA_SVM_+Age,0.664 ± 0.144,0.578 ± 0.155,0.624 ± 0.178,0.715 ± 0.166,0.584 ± 0.226
1,CD_PCA_SVM_+Gender,0.663 ± 0.145,0.578 ± 0.155,0.624 ± 0.178,0.715 ± 0.166,0.584 ± 0.226
2,CD_PCA_SVM_+Both,0.663 ± 0.145,0.578 ± 0.155,0.624 ± 0.178,0.715 ± 0.166,0.584 ± 0.226
3,CD_PCA_LogReg_+Age,0.659 ± 0.147,0.569 ± 0.131,0.613 ± 0.151,0.723 ± 0.148,0.558 ± 0.194
4,CD_PCA_LogReg_+Gender,0.659 ± 0.147,0.569 ± 0.131,0.613 ± 0.151,0.723 ± 0.148,0.558 ± 0.194
5,CD_PCA_LogReg_+Both,0.659 ± 0.147,0.569 ± 0.131,0.613 ± 0.151,0.723 ± 0.148,0.558 ± 0.194
6,CD_KW_SVM_+Age,0.607 ± 0.131,0.613 ± 0.117,0.701 ± 0.205,0.639 ± 0.095,0.844 ± 0.314
7,CD_KW_SVM_+Gender,0.606 ± 0.132,0.618 ± 0.115,0.708 ± 0.199,0.642 ± 0.099,0.852 ± 0.303
8,CD_KW_SVM_+Both,0.605 ± 0.133,0.618 ± 0.115,0.708 ± 0.199,0.642 ± 0.099,0.852 ± 0.303
9,CD_KW_LogReg_+Age,0.599 ± 0.127,0.538 ± 0.123,0.576 ± 0.145,0.704 ± 0.162,0.501 ± 0.157


### 6. Meta-Analysis: Quantifying Metadata Impact (Baseline vs. Multi-Modal)

To conclude our Multi-Modal Integration analysis, we perform an **Ablation Comparison**. We pull the top-performing model from our microbiome-only baseline leaderboard and contrast it side-by-side against our top-performing multi-modal ablation model. 

This explicit comparison isolates the true added value of demographic attributes (Age and Gender) across all five validation metrics.

In [6]:
import pandas as pd
import numpy as np
from IPython.display import display

# 1. Manually input your baseline records from Notebook 05b for a perfect cross-reference
baseline_records = {
    'UC': {
        'Model Configuration': 'UC_PCA_SVM (Microbiome Only)',
        'Auc': '0.578 ± 0.171', 'Accuracy': '0.547 ± 0.168', 
        'F1-Score': '0.512 ± 0.236', 'Precision': '0.556 ± 0.232', 'Recall': '0.500 ± 0.258'
    },
    'CD': {
        'Model Configuration': 'CD_PCA_SVM (Microbiome Only)',
        'Auc': '0.663 ± 0.145', 'Accuracy': '0.578 ± 0.155', 
        'F1-Score': '0.624 ± 0.178', 'Precision': '0.715 ± 0.166', 'Recall': '0.584 ± 0.226'
    }
}

def generate_impact_report(df_ablation, task_key):
    """Aligns the baseline with the top ablation model and calculates the mathematical delta."""
    # Grab the top row from the current notebook's sorted results
    best_multimodal = df_ablation.iloc[0].to_dict()
    
    # Format baseline
    base_row = baseline_records[task_key]
    
    # Combine into a temporary comparison frame
    df_comp = pd.DataFrame([base_row, best_multimodal])
    
    # Calculate absolute delta between the means
    metrics = ['Auc', 'Accuracy', 'F1-Score', 'Precision', 'Recall']
    delta_row = {'Model Configuration': 'Net Change (Δ Mean)'}
    
    for m in metrics:
        mean_base = float(df_comp.iloc[0][m].split(' ')[0])
        mean_multi = float(df_comp.iloc[1][m].split(' ')[0])
        diff = mean_multi - mean_base
        delta_row[m] = f"{diff:+.3f}"
        
    return pd.concat([df_comp, pd.DataFrame([delta_row])], ignore_index=True)

# Generate final tables
report_uc = generate_impact_report(df_uc_combined, 'UC')
report_cd = generate_impact_report(df_cd_combined, 'CD')

print("=================================================================================")
# Note: Casing adjusted to match the exact string headers from your notebook printout
print("📊 FINAL ABLATION ANALYSIS: MICROBIOME VS. MULTI-MODAL")
print("=================================================================================")

print("\n💡 ULCERATIVE COLITIS (UC) HIGHLIGHTS:")
display(report_uc)

print("\n💡 CROHN'S DISEASE (CD) HIGHLIGHTS:")
display(report_cd)

# Export summaries for the paper
report_uc.to_csv("../results/tables/ablation_summary_uc.csv", index=False)
report_cd.to_csv("../results/tables/ablation_summary_cd.csv", index=False)
print("\nSaved summary comparison sheets to '../results/tables/'")

📊 FINAL ABLATION ANALYSIS: MICROBIOME VS. MULTI-MODAL

💡 ULCERATIVE COLITIS (UC) HIGHLIGHTS:


,Model Configuration,Auc,Accuracy,F1-Score,Precision,Recall
0,UC_PCA_SVM (Microbiome Only),0.578 ± 0.171,0.547 ± 0.168,0.512 ± 0.236,0.556 ± 0.232,0.500 ± 0.258
1,UC_PCA_SVM_+Age,0.591 ± 0.148,0.540 ± 0.151,0.491 ± 0.191,0.580 ± 0.145,0.456 ± 0.247
2,Net Change (Δ Mean),+0.013,-0.007,-0.021,+0.024,-0.044



💡 CROHN'S DISEASE (CD) HIGHLIGHTS:


,Model Configuration,Auc,Accuracy,F1-Score,Precision,Recall
0,CD_PCA_SVM (Microbiome Only),0.663 ± 0.145,0.578 ± 0.155,0.624 ± 0.178,0.715 ± 0.166,0.584 ± 0.226
1,CD_PCA_SVM_+Age,0.664 ± 0.144,0.578 ± 0.155,0.624 ± 0.178,0.715 ± 0.166,0.584 ± 0.226
2,Net Change (Δ Mean),+0.001,+0.000,+0.000,+0.000,+0.000



Saved summary comparison sheets to '../results/tables/'
